### 定义工具

In [2]:
import requests

In [ ]:
city = 'Shang hai'
url = f"https://wttr.in/{city}?format=j1"
response = requests.get(url)
response.raise_for_status()
data = response.json()

In [21]:
current_condition = data['current_condition'][0]
output = ''
for k,v in current_condition.items():
    output += f'{k} is {v}, '
output

"FeelsLikeC is -1, FeelsLikeF is 31, cloudcover is 0, humidity is 87, localObsDateTime is 2026-03-31 02:47 AM, observation_time is 12:47 AM, precipInches is 0.0, precipMM is 0.0, pressure is 1023, pressureInches is 30, temp_C is 2, temp_F is 36, uvIndex is 0, visibility is 10, visibilityMiles is 6, weatherCode is 116, weatherDesc is [{'value': 'Partly cloudy'}], weatherIconUrl is [{'value': 'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0004_black_low_cloud.png'}], winddir16Point is NNW, winddirDegree is 328, windspeedKmph is 10, windspeedMiles is 6, "

In [37]:
# 工具函数
def get_weather(
    city:str
):
   try:
     url = f"https://wttr.in/{city}?format=j1"
     response = requests.get(url)
     response.raise_for_status()
     data = response.json()
     current_condition = data['current_condition'][0]
     
     output = '查询到的天气信息如下：'
     weather_desc = current_condition['weatherDesc'][0]['value']
     tempeature_c = current_condition['temp_C']
     output += f"天气描述: {weather_desc}, 温度: {tempeature_c}摄氏度"
     return output
   except requests.exceptions.RequestException as e:
     return f"错误:查询天气遇到网络问题 {e}"
   except (KeyError, IndexError) as e:
        # 处理数据解析错误
     return f"错误:解析天气数据失败，可能是城市名称无效 - {e}"

In [38]:
city = 'New York'
weather = get_weather(city)
weather

'查询到的天气信息如下：天气描述: Partly cloudy, 温度: 2摄氏度'

In [39]:
# 根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
from tavily import TavilyClient
import os
def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    # 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=api_key)
    
    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"
    
    try:
        # 4. 调用API，include_answer=True会返回一个综合性的回答
        response = tavily.search(query=query, search_depth="basic", include_answer=True)
        
        # 5. Tavily返回的结果已经非常干净，可以直接使用
        # response['answer'] 是一个基于所有搜索结果的总结性回答
        if response.get("answer"):
            return response["answer"]
        
        # 如果没有综合性回答，则格式化原始结果
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")
        
        if not formatted_results:
             return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"

In [64]:
# 将所有工具函数放入一个字典，方便后续调用
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}

In [40]:
os.environ["TAVILY_API_KEY"] = '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'
get_attraction(city,weather)

'For partly cloudy, 2°C weather in New York, visit Central Park for a scenic walk or ice skating. The Metropolitan Museum of Art offers indoor cultural enrichment. Go City pass provides discounts on major attractions.'

In [41]:
avaiable_tools = {
    'get_weather':get_weather,
    'get_attraction':get_attraction
}
# 函数映射

### 构建智能体(对话模板)

In [87]:
from openai import OpenAI
class OpenAiChatClient:
    def __init__(
        self,
        model_name:str,
        api_key:str,
        base_url:str = "https://api.minimaxi.com/v1",# minimax
    ):
        self.model_name = model_name
        self.client = OpenAI(api_key = api_key, base_url=base_url)
        
    def generate(
        self,
        prompt:str,
        system_prompt:str
    ) -> str:
        try:
            messages = [
                            {'role': 'system', 'content': system_prompt},
                            {'role': 'user', 'content': prompt}
                    ]
            model = self.model_name
            raw_response = self.client.chat.completions.create(
                model = model,
                messages = messages,
                stream = False,
            )
            response = raw_response.choices[0].message.content
            # print("大语言模型响应成功。")
            return response
        except Exception as e:
            print(f"调用LLM API时发生错误: {e}")
            return "错误:调用语言模型服务时出错。"                

In [88]:
API_KEY ='sk-cp-BrIDPeDEbO4sWRicd2Q_ZNo0_sbE0RgRfun5-mLWTEtRB2W9gIjN8f64Dz8-i8cUKLC1kwK70GPGy2ArtfGcMF_zq4Awn_Y4-ToMqhskLsa0DZpEClrS0Ns'
model_name = 'MiniMax-M2.7'
client = OpenAiChatClient(model_name=model_name,api_key=API_KEY)

In [56]:
system_prompt = '你是一个乐于助人的助手'
prompt = '你是谁？你现在是哪一家的大语言模型？'
response = client.generate(prompt=prompt, system_prompt=system_prompt)

大语言模型响应成功。


In [58]:
response

'<think>\n用户问我我是谁，以及我是哪一家的大语言模型。\n\n根据系统提示，我应该扮演"乐于助人的助手"这个角色。我没有具体说明我是哪个公司或产品的信息。\n\n我应该按照角色设定来回答，表明我是一个乐于助人的AI助手，但不透露我是具体哪家公司的大模型（如果这本身就不是我应该知道的）。\n\n让我以一个友好、简洁的方式回答这个问题。\n</think>\n\n你好！我是你的AI助手，是一个大语言模型。我被设计用来帮助你回答问题、进行对话、提供信息和协助完成各种任务。\n\n有什么我可以帮你的吗？'

### Agent设计

In [59]:
AGENT_SYSTEM_PROMPT = """
你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。

# 可用工具:
- `get_weather(city: str)`: 查询指定城市的实时天气。
- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。

# 输出格式要求:
你的每次回复必须严格遵循以下格式，包含一对Thought和Action：

Thought: [你的思考过程和下一步计划]
Action: [你要执行的具体行动]

Action的格式必须是以下之一：
1. 调用工具：function_name(arg_name="arg_value")
2. 结束任务：Finish[最终答案]

# 重要提示:
- 每次只输出一对Thought-Action
- Action必须在同一行，不要换行
- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[最终答案] 格式结束

请开始吧！
"""

In [92]:
import re
agent = OpenAiChatClient(
    model_name=model_name,
    api_key=API_KEY
)

def main():
    user_prompt = "你好, 请帮助我查询今天广州的天气, 然后根据天气推荐一个合适的旅行计划"
    prompt_history = [f'user prompt: {user_prompt}\n']
    print(f"user prompt: {user_prompt}\n" + "="*40)
    
    for i in range(5):
        print(f"--- 循环 {i+1} ---\n")
        context = "\n".join(prompt_history) # 所有聊天记录拼接在一起
        print(f'上下文:{context}\n'+'='*40)
        response = agent.generate(prompt=context,system_prompt=AGENT_SYSTEM_PROMPT)
        print(f'模型输出:{response}\n')

        # 解析回答
        match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', response, re.DOTALL)
        if match:
            truncated = match.group(1).strip()
            if truncated != response.strip():
                response = truncated
                print("已截断多余的 Thought-Action 对")
        prompt_history.append(response)# 输出Thought:xxx Action:xxx

        # 解析Action
        action_match = re.search(r"Action: (.*)", response, re.DOTALL)
        if not action_match:
            # 错误动作, 直接返回Observation
            observation = "错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ... Action: ...' 的格式。"
            observation_str = f"Observation: {observation}"
            print(f"{observation_str}\n" + "="*40)
            prompt_history.append(observation_str)
            continue
        action_str = action_match.group(1).strip()
        if action_str.startswith("Finish"):
            final_answer = re.match(r"Finish\[(.*)\]", action_str).group(1)
            print(f"任务完成, 最终答案为: {final_answer}")
            break


        # 获得任务参数
        tool_name = re.search(r"(\w+)\(", action_str).group(1)
        args_str = re.search(r"\((.*)\)", action_str).group(1)
        kwargs = dict(re.findall(r'(\w+)="([^"]*)"', args_str))

        # 获得工具和回复
        if tool_name in available_tools:
            observation = available_tools[tool_name](**kwargs)
        else:
            observation = f"错误: 未知的工具 '{tool_name}'。请检查工具名称是否正确，并确保它在可用工具列表中。"

        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)

In [93]:
main()

user prompt: 你好, 请帮助我查询今天广州的天气, 然后根据天气推荐一个合适的旅行计划
--- 循环 1 ---

上下文:user prompt: 你好, 请帮助我查询今天广州的天气, 然后根据天气推荐一个合适的旅行计划

模型输出:<think>
用户想要查询今天广州的天气，然后根据天气推荐一个合适的旅行计划。

我需要先查询广州的天气，获取天气信息后再推荐旅游景点。

让我先调用get_weather函数来查询广州的天气。
</think>

Thought: 用户想要查询今天广州的天气，然后根据天气推荐旅行计划。我需要先调用get_weather工具获取广州的天气信息。
Action: get_weather(city="广州")


已截断多余的 Thought-Action 对
Observation: 查询到的天气信息如下：天气描述: Partly cloudy, 温度: 2摄氏度
--- 循环 2 ---

上下文:user prompt: 你好, 请帮助我查询今天广州的天气, 然后根据天气推荐一个合适的旅行计划

Thought: 用户想要查询今天广州的天气，然后根据天气推荐旅行计划。我需要先调用get_weather工具获取广州的天气信息。
Action: get_weather(city="广州")
Observation: 查询到的天气信息如下：天气描述: Partly cloudy, 温度: 2摄氏度
模型输出:<think>
用户想查询广州的天气，我已经调用了get_weather工具获取到天气信息：多云，气温2摄氏度。

现在我需要根据这个天气信息来推荐旅游景点。我应该调用get_attraction工具，传入城市"广州"和天气"多云"来获取推荐。
</think>

Thought: 已获取到广州今天的天气信息：多云，气温2摄氏度。现在我需要根据这个天气情况调用get_attraction工具来获取适合的旅游景点推荐。
Action: get_attraction(city="广州", weather="多云")


已截断多余的 Thought-Action 对
Observation: 广州多云天气下，广州塔和广东省博物馆是最值得去的景点。广州塔提供观景和娱乐，博物馆展示丰富的文化历史。
--- 循环 3 ---

上